# Reproducing Table 1 of Pele et al. (2026) on S&P 500

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Conformal_Oracle/blob/main/python/examples/notebooks/reproduce_table_1_sp500.ipynb)

This notebook reproduces two rows of **Table 1** (GJR-GARCH and Lag-Llama on S&P 500)
using daily returns from Yahoo Finance, showing the regime classification and Basel
traffic-light zone before and after conformal correction.

**Google Colab:** Run the install cell below first. It will restart the runtime automatically — this is expected. After restart, skip the install cell and run from the imports cell onward.

> On first run, the Lag-Llama model weights (~30 MB) are downloaded from HuggingFace;
> subsequent runs use the cache.

In [ ]:
!pip install -q --upgrade numpy
!pip install -q conformal-oracle yfinance
!pip install -q conformal-oracle[lag_llama]
!pip install -q git+https://github.com/time-series-foundation-models/lag-llama.git

import IPython
IPython.Application.instance().kernel.do_shutdown(True)  # restart runtime

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from conformal_oracle import audit_static, audit_rolling
from conformal_oracle.forecasters import GJRGARCHForecaster, LagLlamaForecaster

np.random.seed(2026)

In [ ]:
spy = yf.download("^GSPC", start="2018-01-01", end="2026-03-13")
returns = np.log(spy["Close"]).diff().dropna().squeeze()
print(f"Loaded {len(returns)} daily log-returns")
returns.tail()

In [ ]:
result_garch = audit_static(
    returns,
    GJRGARCHForecaster(window=250),
    alpha=0.01,
)
print(result_garch.summary())

In [ ]:
try:
    result_lagllama = audit_static(
        returns,
        LagLlamaForecaster(),
        alpha=0.01,
        warmup=512,
    )
    print(result_lagllama.summary())
except ImportError:
    print("Lag-Llama not installed. Install with:")
    print("  pip install conformal-oracle[lag_llama]")
    print("  pip install git+https://github.com/time-series-foundation-models/lag-llama.git")
    result_lagllama = None

## Comparison with paper

| Forecaster | Paper qV | Notebook qV | Paper Regime | Notebook Regime |
|:-----------|:---------|:------------|:-------------|:----------------|
| GJR-GARCH  | −0.005   | *(see above)* | SP         | *(see above)*   |
| Lag-Llama  |  0.008   | *(see above)* | SP         | *(see above)*   |

Fill in the notebook values from the `summary()` output above.
GJR-GARCH should match within ±0.002; Lag-Llama varies with Monte Carlo sampling
but the regime classification (signal-preserving) should be invariant.

In [ ]:
result_garch_roll = audit_rolling(
    returns,
    GJRGARCHForecaster(window=250),
    alpha=0.01,
    window=250,
)
print(result_garch_roll.summary())

## Notes

- Numerical differences from the paper reflect Monte Carlo variation in TSFM
  sampling (Lag-Llama uses 1000 samples per forecast by default).
- Different historical data windows may shift qV estimates slightly; the regime
  classification (signal-preserving vs replacement) is robust.
- Full paper replication requires running the panel inference via
  `conformal_oracle.panel.audit_panel` on all 24 assets.
- See the [methodology docs](../../docs/methodology.md) and the
  [API reference](../../docs/api.md) for details.